# detach-clone-snapshot — faded example 2: Fill the snapshot that survives an in-place mutation

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`. The last cell reports your progress on the `PyTorch: detach + clone snapshot` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: detach + clone snapshot` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-clone-snapshot`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-clone-snapshot"
DD_SUBTOPIC = "PyTorch: detach + clone snapshot"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`x.detach()` returns a graph-free tensor that still SHARES storage with `x`, so an in-place write to `x` is visible through it. `x.detach().clone()` additionally copies the storage, so the clone is frozen at the moment of snapshotting. The `.clone()` is the load-bearing operation for storage independence.

## Faded exercise 2

### Complete the storage-independent snapshot

We take a graph-free view `shared = x.detach()` and a true snapshot `snap`, then mutate `x` in place via `x.data[0] = 50.0`. Complete the line defining `snap` so that after the mutation `shared[0]` reflects the new value (50.0) but `snap[0]` still holds the original value of `x[0]`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
t.manual_seed(0)

def detach_vs_clone(x):
    shared = x.detach()
    snap = None  # TODO: fill in this step — read the prompt cell above
    x.data[0] = 50.0
    return {
        'shared_first': float(shared[0]),
        'snap_first': float(snap[0]),
        'snap_shares_storage': snap.data_ptr() == x.data_ptr(),
        'snap_requires_grad': bool(snap.requires_grad),
    }

x = t.tensor([2.0, 7.0, 1.0], requires_grad=True)
print(detach_vs_clone(x))

def _test():
    x = t.tensor([2.0, 7.0, 1.0], requires_grad=True)
    orig_first = float(x[0])
    out = detach_vs_clone(x)
    assert out['shared_first'] == 50.0, out['shared_first']
    assert out['snap_first'] == orig_first, (out['snap_first'], orig_first)
    assert out['snap_shares_storage'] is False
    assert out['snap_requires_grad'] is False

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

def detach_vs_clone(x):
    shared = x.detach()
    snap = x.detach().clone()
    x.data[0] = 50.0
    return {
        'shared_first': float(shared[0]),
        'snap_first': float(snap[0]),
        'snap_shares_storage': snap.data_ptr() == x.data_ptr(),
        'snap_requires_grad': bool(snap.requires_grad),
    }

x = t.tensor([2.0, 7.0, 1.0], requires_grad=True)
print(detach_vs_clone(x))
```
</details>